Some info about the data:
- the neural data is recorded at 30000 Hz
- the behavioral data is recorded at 40 Hz (we refer to the behavioral sampling times as frames)
- threatening stimuli are usually an air puff delivered simultaneously with an auditory threat (sometimes it's only an auditory threat). The threats are always delivered in the same place at exactly the opposite end of the arena compared to the shelter.
- The video data is collected at 1024x1024 pixels (~10pixels per cm). All positional information is in pixels.

In [ ]:
import numpy as np
import os
import polars as pl
import dill as pickle
import socket
import tkinter as tk
from tkinter import filedialog

In [ ]:
"""The paths to all the sessions"""
# base path: identifies where you have mapped ceph onto your computer
# all_paths: a list of the paths to specific experimental sessions 
# NB: some paths are commented out! This is because we are migrating some of our data between servers right now. It should be available on ceph in a few days.

def get_computer_specific_paths():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    base_path = filedialog.askdirectory(title="Select Directory in which you have mounted Branco lab ceph")
    return base_path

base_path = get_computer_specific_paths()

all_paths = ['Jasmine_Laurence/Experimental_Data/JAL004/004_baseline_2023_08_17T13_41_44',
             'Jasmine_Laurence/Experimental_Data/JAL004/JAL004_flip_rotated_2023_08_28T09_36_04',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flip_2023_09_03T12_04_16',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flip_puff2_2023_09_11T09_32_25',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flipppuf19sept_2023_09_19T14_10_56',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_baseline_2023_09_05T07_48_58',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_flip1_2023_09_08T07_36_54',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_flippuff3_2023_09_21T11_11_13',
             'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_empty_shelter_2024_03_04T11_24_29',
             'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_barrier_flip3_2024_03_18T11_53_29',
             'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34',
             'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33',
             'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_empty_shelter_2024_03_05T13_45_47',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_barrierflip2_2024_03_12T11_18_26',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_5_2024_03_22T11_15_43',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_8_2024_04_09T10_07_45',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_9_2024_04_16T11_13_05',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_100_2024_04_23T09_59_40',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_tinnybarrier1_2024_04_30T10_57_04',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_empty_shelter2_2024_04_22T10_51_22',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_1_2024_04_25T11_27_42',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_tiny_barrier_flip_1_2024_05_03T10_02_35',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_5_2024_05_14T10_18_03',
             'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_tiny_flip_2_2024_05_21T11_10_19'
             ]


In [ ]:
"""Choose an experiment to load"""

idx = 0
experiment_path = all_paths[idx]

In [ ]:
"""Get data paths and session metadata for a given session"""
# session object has a lot of information about the recording. The following could be useful to you:
# session.audio.onset_frames is a list (sometimes it's a list of lists sorry!) of times in behavioral frames of when the threat was delivered
# session.shelter_location gives you the xy position of the top left and bottom right corners of the shelter
# session.barrier_time gives you the time in minutes when the barrier was first placed in the arena (you probably want everything before this time)

with open(os.path.join(base_path, experiment_path, "processed_data", "metadata"), "rb") as dill_file: 
    session = pickle.load(dill_file)

In [ ]:
""""Load time x neurons matrix"""
# this will give you the firing rate of each putative single unit at each frame (behavioral sampling timepoint)

frame_by_cluster_matrix = np.load(os.path.join(base_path, experiment_path, "processed_data") + "\\" + "frame_by_good_cluster_matrix.npy")

In [ ]:
"""Load spikes dataframe
An alternative way to load in neural data that gives you individual spike times for both single and multiunit clusters
"""

# loads in a polars dataframe with the following columns
# spike_df['aligned_spike_times']: IGNORE (spike times in behavioral computer clock)
# spike_df['spike_aligned_to_frame']: the frame (behavioral sampling timepoint) that a given spike was recorded on
# spike_df['aligned_spike_times_in_samples']: IGNORE (spike times in behavioral computer clock)
# spike_df['spike_times']: IGNORE  (spike time in recording computer clock)
# spike_df['spike_clusters']: the cluster that a given spike belongs to
# spike_df['cluster_group']: the spikesorting classification of a given cluster as 'good' (putative single unit), 'mua' (multiunit activity), 'noise' (noise cluster)

spike_df = pl.read_csv(os.path.join(base_path, experiment_path, "processed_data", "Processed_efizz_data.csv"))

In [ ]:
"""Load behavioral dataframe"""

# These are the columns of the dataframe
# -- frames: the behavioral frame number (this will match the rows of the time x neurons matrix and the frames in spike_df['spike_aligned_to_frame'])
# -- hdir: the mouse's head direction 
# -- hsa: the mouse's head-shelter angle
# -- mouse_x_position
# -- mouse_y_position
# -- OutofshelterIdx (bool): If True the mouse was outside the shelter
# -- EscapePeriod (bool): If True the mouse is performing an escape
# -- shelter (bool): If True the shelter is present in the arena
# -- barrier_present (bool): If True the barrier is present in the arena (to only look at times when there is no barrier you can use this to fin the frames before this becomes True)
# -- barrier_flipped (bool): If True the barrier has been flipped 180deg
# -- speed
# -- homingPeriod (bool): If True the mouse is performing a homing run
# -- h_preflipbar_a: IGNORE: the angle between the mouse's head and the open side of the barrier before flipping it
# -- h_postflipbar_a: IGNORE: the angle between the mouse's head and the open side of the barrier after flipping it
# -- h_bar_centre_a: IGNORE: the angle between the mouse's head and the centre of the barrier (also the centre of the arena)

video_df = pl.read_csv(os.path.join(base_path, experiment_path, "processed_data", "full_video_dataframe.csv"))